<a href="https://colab.research.google.com/github/g00dwill47ch0ppas/Image-Classification-ITRI626-Project/blob/tshifhiwa/PlantVillage_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ITRI626 Project — PlantVillage Plant Disease Classification

**Group project notebook**: Custom CNN vs ResNet50 (transfer learning) vs Vision Transformer (transfer learning)

Structure:
1. Setup and dataset loading
2. Data inspection and cleaning (duplicate detection)
3. Train / validation / test split (leakage-safe)
4. Preprocessing and augmentation
5. Model 1: Custom CNN baseline
6. Model 2: ResNet50 (transfer learning)
7. Model 3: Vision Transformer (transfer learning)
8. Evaluation and comparison
9. Error analysis

> Fill in every `TODO` before submission. Re-run top to bottom before saving final outputs so the notebook shows a clean, reproducible run.

## 0. Setup

Run this first. If using Colab, make sure Runtime → Change runtime type → GPU is selected.

In [17]:
#!pip install timm imagehash -q  # uncomment on first run

import os, random, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cuda


### Mount Drive / get the dataset

Option A — dataset already unzipped in Drive:

In [18]:
#from google.colab import drive
#drive.mount('/content/drive')

# TODO: update this path to wherever you unzipped PlantVillage
#DATA_ROOT = "/content/drive/MyDrive/PlantVillage"

Option B — pull directly from Kaggle (uncomment if you'd rather not store the unzipped dataset in Drive):

In [ ]:
from google.colab import files
files.upload()  # upload your kaggle.json API token here
!mkdir -p ~/.kaggle && echo KGAT_bf759ab85f815bad91cadffb83842665 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
!pip install kaggle -q
!kaggle datasets download -d emmarex/plantdisease -p /content/plantvillage --unzip
DATA_ROOT = "/content/plantvillage/PlantVillage"  # adjust to actual extracted folder name


In [22]:
import os

if os.path.exists('/content/plantvillage'):
    print(os.listdir('/content/plantvillage'))
else:
    print('/content/plantvillage does not exist. Please ensure the dataset is downloaded or unzipped correctly.')

/content/plantvillage does not exist. Please ensure the dataset is downloaded or unzipped correctly.


## 1. Data inspection

Count images per class before any cleaning. This table goes directly into your report.

In [21]:
class_names = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f"{len(class_names)} classes found\n")

raw_counts = {}
for cls in class_names:
    cls_path = os.path.join(DATA_ROOT, cls)
    n = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    raw_counts[cls] = n
    print(f"{cls}: {n}")

print(f"\nTotal images (before cleaning): {sum(raw_counts.values())}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/plantvillage/PlantVillage'

In [ ]:
# Bar chart of class counts - include this figure in your report
plt.figure(figsize=(12, 6))
plt.bar(range(len(raw_counts)), list(raw_counts.values()))
plt.xticks(range(len(raw_counts)), list(raw_counts.keys()), rotation=90)
plt.ylabel("Image count")
plt.title("PlantVillage: images per class (before cleaning)")
plt.tight_layout()
plt.savefig("class_distribution_before.png", dpi=150)
plt.show()


Show a few representative examples from every class (required by the rubric):

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))  # TODO: adjust grid to your class count
sample_classes = class_names[:20]  # TODO: show all classes if manageable, else a representative subset
for ax, cls in zip(axes.flat, sample_classes):
    cls_path = os.path.join(DATA_ROOT, cls)
    # Find the first image file in the directory
    fname = None
    for item in os.listdir(cls_path):
        full_path = os.path.join(cls_path, item)
        if os.path.isfile(full_path) and item.lower().endswith(('.jpg', '.jpeg', '.png')):
            fname = item
            break

    if fname is None:
        # If no image found, skip this class or handle appropriately
        ax.set_title(f"{cls}\n(No image)", fontsize=8, color='red')
        ax.axis("off")
        continue

    img = Image.open(os.path.join(cls_path, fname))
    ax.imshow(img)
    ax.set_title(cls, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig("class_examples.png", dpi=150)
plt.show()

## 2. Cleaning: corrupted files and duplicate detection

PlantVillage is known to contain near-duplicate images (same source leaf, cropped/rotated differently). We use perceptual hashing to group these so we can prevent them being split across train/test later.

In [ ]:
import imagehash
from collections import defaultdict

hash_to_files = defaultdict(list)
corrupted = []

for cls in class_names:
    cls_path = os.path.join(DATA_ROOT, cls)
    for fname in os.listdir(cls_path):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(cls_path, fname)
        try:
            h = imagehash.phash(Image.open(fpath))
            hash_to_files[str(h)].append((cls, fname))
        except Exception as e:
            corrupted.append(fpath)

print(f"Corrupted/unreadable files found: {len(corrupted)}")
for f in corrupted:
    print(" -", f)

duplicate_groups = {h: files for h, files in hash_to_files.items() if len(files) > 1}
n_dup_images = sum(len(files) for files in duplicate_groups.values())
print(f"\n{len(duplicate_groups)} duplicate/near-duplicate groups")
print(f"{n_dup_images} images involved in duplicate groups (these will be kept together in the same split)")


**Note for report:** we don't necessarily delete near-duplicates — the safer approach is to keep every duplicate *group* entirely within one split (train, val, or test), so no leakage occurs. Document this decision explicitly. If you'd rather drop duplicates entirely, do that here and record how many images were removed and why.

## 3. Train / validation / test split (leakage-safe, stratified, seeded)

Each duplicate group is treated as one unit and assigned entirely to one split.

In [ ]:
groups = list(hash_to_files.items())  # [(hash, [(cls, fname), ...]), ...]
group_labels = [files[0][0] for _, files in groups]

train_groups, temp_groups, train_labels, temp_labels = train_test_split(
    groups, group_labels, test_size=0.30, stratify=group_labels, random_state=SEED
)
val_groups, test_groups, _, _ = train_test_split(
    temp_groups, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)

def flatten(groups):
    files = []
    for _, group_files in groups:
        files.extend(group_files)
    return files

train_files, val_files, test_files = flatten(train_groups), flatten(val_groups), flatten(test_groups)

print(f"Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}")
print(f"Split ratio: {len(train_files)/n_dup_images*100 if False else ''}")

# Class distribution per split - include this table in the report
from collections import Counter
for name, files in [("Train", train_files), ("Val", val_files), ("Test", test_files)]:
    counts = Counter(cls for cls, _ in files)
    print(f"\n{name} class distribution:")
    for cls in class_names:
        print(f"  {cls}: {counts.get(cls, 0)}")


Save the split so it's reproducible and can be submitted alongside the notebook (required submission item):

In [ ]:
split_record = {
    "seed": SEED,
    "train": [f"{c}/{f}" for c, f in train_files],
    "val": [f"{c}/{f}" for c, f in val_files],
    "test": [f"{c}/{f}" for c, f in test_files],
}
with open("split.json", "w") as fh:
    json.dump(split_record, fh)
print("Saved split.json")


## 4. Preprocessing, augmentation, and Dataset/DataLoader

Augmentations are applied to training data only, and are chosen to be realistic for leaf photos (flips, small rotations, mild colour jitter) — not distortions that wouldn't occur naturally.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32  # TODO: lower to 16 if you hit GPU memory limits, especially for ViT

class_to_idx = {c: i for i, c in enumerate(class_names)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class PlantDataset(Dataset):
    def __init__(self, files, root, transform):
        self.files = files
        self.root = root
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        cls, fname = self.files[idx]
        img = Image.open(os.path.join(self.root, cls, fname)).convert("RGB")
        return self.transform(img), class_to_idx[cls]

train_ds = PlantDataset(train_files, DATA_ROOT, train_tf)
val_ds = PlantDataset(val_files, DATA_ROOT, eval_tf)
test_ds = PlantDataset(test_files, DATA_ROOT, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2)

print(f"Batches - train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")


## 5. Shared training and evaluation functions

Reused identically for all three models so the comparison is fair.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-4, weight_decay=1e-4, patience=4, name="model"):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        train_loss = total_loss / len(train_loader)
        history["train_loss"].append(train_loss)

        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                val_loss += criterion(out, y).item()
                correct += (out.argmax(1) == y).sum().item()
                total += y.size(0)
        val_loss /= len(val_loader)
        val_acc = correct / total
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        scheduler.step()

        print(f"[{name}] Epoch {epoch+1}/{epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{name}_best.pt")  # checkpoint - protects against Colab disconnects
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(f"{name}_best.pt"))
    return model, history


def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            out = model(x)
            preds = out.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_preds)


def plot_curves(history, name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train_loss")
    axes[0].plot(history["val_loss"], label="val_loss")
    axes[0].set_title(f"{name}: Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history["val_acc"], label="val_acc", color="green")
    axes[1].set_title(f"{name}: Validation Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.savefig(f"{name}_curves.png", dpi=150)
    plt.show()


## 6. Model 1: Custom CNN baseline

No pretrained weights — this is your transparent, fully-explainable baseline.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

num_classes = len(class_names)
model_cnn = SimpleCNN(num_classes)
model_cnn, history_cnn = train_model(model_cnn, train_loader, val_loader, epochs=20, lr=1e-3, name="cnn")


In [ ]:
plot_curves(history_cnn, "Custom CNN")

## 7. Model 2: ResNet50 (transfer learning)

In [ ]:
model_resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, num_classes)

model_resnet, history_resnet = train_model(model_resnet, train_loader, val_loader, epochs=15, lr=1e-4, name="resnet50")


In [ ]:
plot_curves(history_resnet, "ResNet50")

## 8. Model 3: Vision Transformer (transfer learning)

Uses `timm` for the pretrained ViT. If training is slow on your GPU quota, switch to `vit_small_patch16_224` or lower `IMG_SIZE`/`BATCH_SIZE` — it's still the same model family for the rubric.

In [ ]:
import timm

model_vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=num_classes)

model_vit, history_vit = train_model(model_vit, train_loader, val_loader, epochs=15, lr=5e-5, name="vit")


In [ ]:
plot_curves(history_vit, "Vision Transformer")

## 9. Test-set evaluation and comparison

All three models evaluated on the exact same held-out test set.

In [ ]:
results = {}
for name, model in [("Custom CNN", model_cnn), ("ResNet50", model_resnet), ("ViT", model_vit)]:
    y_true, y_pred = evaluate_model(model, test_loader)
    acc = (y_true == y_pred).mean()
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    results[name] = {"y_true": y_true, "y_pred": y_pred, "accuracy": acc, "macro_f1": macro_f1}
    print(f"\n=== {name} ===")
    print(f"Test accuracy: {acc:.4f}   Macro F1: {macro_f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


In [ ]:
# Comparison summary table
import pandas as pd
summary = pd.DataFrame({
    name: {"Test Accuracy": r["accuracy"], "Macro F1": r["macro_f1"]}
    for name, r in results.items()
}).T
print(summary)
summary.to_csv("model_comparison_summary.csv")


In [ ]:
# Confusion matrices - required figure, one per model
import seaborn as sns

for name, r in results.items():
    cm = confusion_matrix(r["y_true"], r["y_pred"])
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix: {name}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=90); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{name.replace(' ', '_')}.png", dpi=150)
    plt.show()


## 10. Error analysis

Required by the rubric: show several correct and incorrect predictions, and discuss patterns in the mistakes.

In [ ]:
def show_predictions(model, dataset, n=8, name="model"):
    model.eval()
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    shown = 0
    with torch.no_grad():
        for i in range(len(dataset)):
            if shown >= n:
                break
            x, y = dataset[i]
            out = model(x.unsqueeze(0).to(DEVICE))
            pred = out.argmax(1).item()
            ax = axes.flat[shown]
            img_display = x.permute(1, 2, 0).numpy()
            img_display = (img_display * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)).clip(0, 1)
            ax.imshow(img_display)
            correct = pred == y
            ax.set_title(f"True: {class_names[y]}\nPred: {class_names[pred]}",
                         color="green" if correct else "red", fontsize=9)
            ax.axis("off")
            shown += 1
    plt.suptitle(f"{name}: sample predictions")
    plt.tight_layout()
    plt.savefig(f"{name}_sample_predictions.png", dpi=150)
    plt.show()

# TODO: for a more useful figure, deliberately select some correct AND some incorrect examples
# rather than just the first N images - e.g. filter results[name]['y_true'] != results[name]['y_pred']
show_predictions(model_resnet, test_ds, n=8, name="ResNet50")


**TODO in your report/discussion:**
- Which classes are most confused with each other, and does that make biological sense (e.g. visually similar disease symptoms)?
- Does performance differ for underrepresented classes (check per-class F1 from the classification reports above)?
- Which model generalized best, and why might that be (data efficiency, pretrained features, model capacity)?
- Any limitations: image quality, class imbalance, dataset scope (lab photos vs field photos), etc.

## 11. Reproducibility notes (for README / report)

Fill in the actual values used once your final run is locked in:

| Item | Value |
|---|---|
| Dataset | PlantVillage (source: TODO link, licence: TODO) |
| Classes used | TODO |
| Random seed | 42 |
| Split | 70/15/15, leakage-safe by duplicate group, stratified |
| Input size | 224x224 |
| Batch size | 32 (TODO if changed) |
| Optimizer | AdamW |
| LR schedule | Cosine annealing |
| Loss | CrossEntropyLoss |
| Framework | PyTorch + torchvision + timm |
| Hardware | TODO: GPU model from Colab (Runtime > View resources) |